In [4]:
import json
import yfinance as yf
from transformers import pipeline
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import numpy as np

print(" imports done")

 imports done


In [5]:
# LIVE MARKET DATA

@tool
def get_market_data(ticker: str) -> str:
    """
    Fetch live market data for a stock, index or commodity.
    Use for current price, P/E ratio, market cap, 52w high/low.
    Examples: AAPL, RELIANCE.NS, GC=F (gold), CL=F (oil), ^NSEI
    """
    try:
        info = yf.Ticker(ticker.upper()).info
        data = {
            "ticker":         ticker.upper(),
            "name":           info.get("longName", "N/A"),
            "price":          info.get("currentPrice") or info.get("regularMarketPrice"),
            "currency":       info.get("currency", "USD"),
            "pe_ratio":       info.get("trailingPE"),
            "market_cap":     info.get("marketCap"),
            "52w_high":       info.get("fiftyTwoWeekHigh"),
            "52w_low":        info.get("fiftyTwoWeekLow"),
            "analyst_rating": info.get("recommendationKey", "N/A"),
            "sector":         info.get("sector", "N/A"),
        }
        return json.dumps(data, indent=2, default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})

# Test
print(get_market_data.invoke("AAPL"))

{
  "ticker": "AAPL",
  "name": "Apple Inc.",
  "price": 308.63,
  "currency": "USD",
  "pe_ratio": 37.319225,
  "market_cap": 4532958920704,
  "52w_high": 317.4,
  "52w_low": 201.5,
  "analyst_rating": "buy",
  "sector": "Technology"
}


In [6]:
import torch
import json
import yfinance as yf
from langchain_core.tools import tool

_finbert = None

def load_finbert():
    global _finbert
    if _finbert is None:
        print("Loading FinBERT...")
        from transformers import pipeline as hf_pipeline
        _finbert = hf_pipeline(
            "text-classification",
            model="ProsusAI/finbert",
            top_k=1,
            truncation=True,
        )
        print("FinBERT ready")
    return _finbert


@tool
def get_news_sentiment(ticker: str) -> str:
    """
    Get recent news headlines and their sentiment using FinBERT.
    Use for: is market mood bullish or bearish for this asset?
    Examples: AAPL, gold, HDFC, oil
    """
    try:
        headlines = [
            a["title"] for a in yf.Ticker(ticker).news[:8]
            if a.get("title")
        ]
        if not headlines:
            return f"No news found for {ticker}"

        pipe    = load_finbert()
        counts  = {"positive": 0, "negative": 0, "neutral": 0}
        results = []

        for h in headlines:
            label = pipe(h)[0][0]["label"].lower()
            counts[label] += 1
            results.append({"headline": h, "sentiment": label})

        total   = len(headlines)
        overall = (
            "BULLISH" if counts["positive"]/total >= 0.6 else
            "BEARISH" if counts["negative"]/total >= 0.6 else
            "NEUTRAL"
        )

        return json.dumps({
            "overall":  overall,
            "counts":   counts,
            "articles": results
        }, indent=2)

    except Exception as e:
        return f"Error: {e}"

# test
print(get_news_sentiment.invoke("AAPL"))

No news found for AAPL


In [7]:
#FINANCIAL CALCULATOR

@tool
def financial_calculator(calculation_type: str, parameters: str) -> str:
    """
    Perform financial calculations.
    Types: dcf, sharpe, cagr, compound_interest
    Pass parameters as a JSON string.
    DCF:      {"fcfs":[100,120,140], "discount_rate":0.10, "terminal_growth":0.03}
    Sharpe:   {"returns":[0.05,-0.02,0.08], "risk_free_rate":0.04}
    CAGR:     {"start_value":50000, "end_value":120000, "years":8}
    CI:       {"principal":100000, "annual_rate":0.08, "years":10, "n":12}
    """
    try:
        p    = json.loads(parameters)
        calc = calculation_type.lower()

        if calc == "dcf":
            fcfs = p["fcfs"]
            r    = p.get("discount_rate", 0.10)
            g    = p.get("terminal_growth", 0.03)
            pvs  = [fcf / (1 + r) ** t for t, fcf in enumerate(fcfs, 1)]
            tv   = fcfs[-1] * (1 + g) / (r - g)
            pv_tv = tv / (1 + r) ** len(fcfs)
            return json.dumps({
                "pv_cashflows":  round(sum(pvs), 2),
                "terminal_value": round(tv, 2),
                "intrinsic_value": round(sum(pvs) + pv_tv, 2),
            }, indent=2)

        elif calc == "sharpe":
            rets = np.array(p["returns"])
            rf   = p.get("risk_free_rate", 0.04)
            sr   = (np.mean(rets) - rf) / np.std(rets, ddof=1)
            return json.dumps({
                "sharpe_ratio":   round(float(sr), 4),
                "interpretation": "Excellent" if sr > 1 else "Good" if sr > 0.5 else "Poor",
            }, indent=2)

        elif calc == "cagr":
            cagr = (p["end_value"] / p["start_value"]) ** (1 / p["years"]) - 1
            return json.dumps({"cagr_pct": round(cagr * 100, 2)}, indent=2)

        elif calc == "compound_interest":
            P, r, n, t = p["principal"], p["annual_rate"], p.get("n", 12), p["years"]
            A = P * (1 + r / n) ** (n * t)
            return json.dumps({
                "future_value":   round(A, 2),
                "interest_earned": round(A - P, 2),
            }, indent=2)

        else:
            return json.dumps({"error": f"Unknown type: {calc}"})

    except Exception as e:
        return json.dumps({"error": str(e)})

# Test
print(financial_calculator.invoke({
    "calculation_type": "cagr",
    "parameters": '{"start_value": 50000, "end_value": 120000, "years": 8}'
}))

{
  "cagr_pct": 11.56
}


In [8]:
#CONCEPT EXPLAINER

_concept_llm   = None
_concept_chain = None

def get_concept_chain():
    global _concept_llm, _concept_chain
    if _concept_chain is None:
        _concept_llm = ChatOllama(model="llama3.2", temperature=0.3)
        prompt = ChatPromptTemplate.from_template("""
You are a financial literacy teacher.
Explain the concept below in simple, clear language.
Adjust depth: start simple, then give a practical example.
Avoid jargon. If jargon is needed, define it immediately.

Concept: {concept}

Explanation:
""")
        _concept_chain = prompt | _concept_llm | StrOutputParser()
    return _concept_chain


@tool
def explain_concept(concept: str) -> str:
    """
    Explain a financial concept in simple language with examples.
    Use when user asks what a term means or seems unfamiliar with it.
    Examples: P/E ratio, volatility, diversification, beta, CAGR,
              market cap, dividend yield, inflation, recession
    """
    try:
        chain = get_concept_chain()
        return chain.invoke({"concept": concept})
    except Exception as e:
        return f"Could not explain concept: {e}"

# Test
print(explain_concept.invoke("P/E ratio"))  

Welcome to our lesson on financial literacy! Today, we're going to talk about the Price-to-Earnings (P/E) ratio.

**What is the P/E ratio?**

The P/E ratio is a way to compare the price of a stock to its earnings. Think of it like this: if you buy a stock for $100 and it makes $10 in profits, how much are you paying per dollar of profit? That's basically what the P/E ratio tells you.

**Simple Example**

Let's say you own 10 shares of a company called "Tech Inc." The price of each share is $20. If Tech Inc. made $100 in profits last year, the P/E ratio would be:

P/E Ratio = Price per Share ÷ Earnings per Share
= $20 ÷ $10
= 2

This means that for every dollar of profit, you're paying $2.

**Practical Example**

Now, let's say you want to buy more shares of Tech Inc. The price has gone up to $25 per share, and the company still made $100 in profits last year. What does this mean for the P/E ratio?

P/E Ratio = Price per Share ÷ Earnings per Share
= $25 ÷ $10
= 2.5

In this case, you're

In [9]:
all_tools = [
    get_market_data,
    get_news_sentiment,
    financial_calculator,
    explain_concept,
]

print(f"Total tools: {len(all_tools)}")
for t in all_tools:
    print(f"  - {t.name}")

Total tools: 4
  - get_market_data
  - get_news_sentiment
  - financial_calculator
  - explain_concept


In [10]:
# Market data
print("MARKET DATA")
print(get_market_data.invoke("GC=F"))

# Calculator
print("\n SHARPE RATIO ")
print(financial_calculator.invoke({
    "calculation_type": "sharpe",
    "parameters": '{"returns": [0.05, -0.02, 0.08, 0.03], "risk_free_rate": 0.04}'
}))

# Concept
print("\n CONCEPT EXPLAINER ")
print(explain_concept.invoke("diversification"))

MARKET DATA
{
  "ticker": "GC=F",
  "name": "N/A",
  "price": 4187.3,
  "currency": "USD",
  "pe_ratio": null,
  "market_cap": null,
  "52w_high": 5586.2,
  "52w_low": 3263.9,
  "analyst_rating": "N/A",
  "sector": "N/A"
}

 SHARPE RATIO 
{
  "sharpe_ratio": -0.119,
  "interpretation": "Poor"
}

 CONCEPT EXPLAINER 
Welcome to our lesson on diversification! Today, we're going to talk about how to spread your investments across different types of assets to reduce risk and increase potential returns.

**What is diversification?**

Imagine you have a garden with only one type of flower. If the weather gets bad, all your flowers might die. But if you plant different types of flowers in your garden, like roses, daisies, and sunflowers, they'll be less affected by bad weather. That's kind of like what diversification does for your investments.

**Why is diversification important?**

When you put all your money into one type of investment, it's like putting all your eggs in one basket. If that